In [1]:
pip install xgboost --no-binary xgboost -v


Using pip 26.0.1 from /Users/temuulenbadarchoutlook.com/Library/Python/3.9/lib/python/site-packages/pip (python 3.9)
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [13]:
import sys

# Upgrade pip first
!{sys.executable} -m ensurepip --upgrade
!{sys.executable} -m pip install --upgrade pip

# Install XGBoost cleanly
!{sys.executable} -m pip install --upgrade xgboost

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /var/folders/fb/60df8q8s02xdfy1j644kdxg00000gn/T/tmpppocko_2
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [2]:
pip install ccxt pandas numpy xgboost ta scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
import ccxt
import pandas as pd
import numpy as np
import time

from datetime import datetime
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from ta.momentum import RSIIndicator
from ta.trend import MACD, SMAIndicator
from ta.volatility import BollingerBands
from ta.volume import VolumeWeightedAveragePrice

In [ ]:
# ================================
# LIVE PRICE STREAM (1 second)
# ================================

import ccxt
import time
from datetime import datetime

SYMBOL = "BTC/USDT"

exchange = ccxt.binance()

print("Starting live BTC price stream...")

while True:
    try:
        ticker = exchange.fetch_ticker(SYMBOL)

        price = ticker["last"]
        timestamp = datetime.utcnow()

        print(f"{price}")

        time.sleep(1)

    except Exception as e:
        print("Error:", e)
        time.sleep(2)

Starting live BTC price stream...
2026-03-07 06:50:08.349807 |    67892.63
2026-03-07 06:50:09.541488 |    67892.63
2026-03-07 06:50:10.735046 |    67895.14
2026-03-07 06:50:11.925839 |    67895.91
2026-03-07 06:50:13.117082 |    67895.92
2026-03-07 06:50:14.309085 |    67895.92
2026-03-07 06:50:15.497695 |    67895.92
2026-03-07 06:50:16.690648 |    67899.99
2026-03-07 06:50:17.879383 |    67899.98
2026-03-07 06:50:19.068274 |    67899.99
2026-03-07 06:50:20.262594 |    67899.99
2026-03-07 06:50:21.455895 |    67892.63
2026-03-07 06:50:22.648883 |    67842.37
2026-03-07 06:50:23.862954 |    67842.37
2026-03-07 06:50:25.062718 |    67842.37
2026-03-07 06:50:26.253606 |    67842.37
2026-03-07 06:50:27.445035 |    67842.38
2026-03-07 06:50:28.646575 |    67833.37
2026-03-07 06:50:29.846242 |    67830.01
2026-03-07 06:50:31.043190 |    67830.01
2026-03-07 06:50:32.235503 |    67820.02
2026-03-07 06:50:33.434070 |    67820.02
2026-03-07 06:50:34.624246 |    67824.26
2026-03-07 06:50:35.816

In [1]:
# ================================
# CRYPTO ML LIVE PREDICTION ENGINE
# Retrains every 1 minute
# ================================

# INSTALL (run once if needed)
# pip install ccxt pandas numpy xgboost ta scikit-learn matplotlib seaborn

# ================================
# IMPORTS
# ================================
import ccxt
import pandas as pd
import numpy as np
import time
from datetime import datetime
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from sklearn.calibration import CalibratedClassifierCV
from ta.trend import SMAIndicator, MACD
from ta.momentum import RSIIndicator
from ta.volatility import BollingerBands
from ta.volume import VolumeWeightedAveragePrice

# ================================
# CONFIG
# ================================
SYMBOL = "BTC/USDT"
TIMEFRAME = "1m"
CANDLE_LIMIT = 2000      # keep last N candles
TARGET_LOOKAHEAD = 1     # predict next minute
EXCHANGE = ccxt.binance()

FEATURES = [
    "return_1m_smooth",
    "return_5m",
    "volatility_smooth",
    "volume_change",
    "sma10",
    "sma20",
    "momentum_smooth",
    "rsi",
    "macd",
    "bb_width",
    "vwap"
]

# ================================
# FETCH MARKET DATA
# ================================
def fetch_data(limit=CANDLE_LIMIT):
    ohlcv = EXCHANGE.fetch_ohlcv(SYMBOL, timeframe=TIMEFRAME, limit=limit)
    df = pd.DataFrame(ohlcv, columns=["timestamp","open","high","low","close","volume"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    return df

def fetch_latest_candle():
    ohlcv = EXCHANGE.fetch_ohlcv(SYMBOL, timeframe=TIMEFRAME, limit=2)
    latest = ohlcv[-1]
    df = pd.DataFrame([latest], columns=["timestamp","open","high","low","close","volume"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    return df

# ================================
# FEATURE ENGINEERING
# ================================
def build_features(df):
    df["return_1m"] = df["close"].pct_change()
    df["return_1m_smooth"] = df["return_1m"].rolling(3).mean()
    df["return_5m"] = df["close"].pct_change(5)

    df["volatility"] = df["return_1m"].rolling(10).std()
    df["volatility_smooth"] = df["volatility"].rolling(3).mean()

    df["volume_change"] = df["volume"].pct_change()

    df["sma10"] = SMAIndicator(df["close"], window=10).sma_indicator()
    df["sma20"] = SMAIndicator(df["close"], window=20).sma_indicator()

    df["momentum"] = df["close"] - df["close"].shift(10)
    df["momentum_smooth"] = df["momentum"].rolling(3).mean()

    df["rsi"] = RSIIndicator(df["close"], window=14).rsi()
    df["macd"] = MACD(df["close"]).macd()
    bb = BollingerBands(df["close"])
    df["bb_width"] = bb.bollinger_hband() - bb.bollinger_lband()
    df["vwap"] = VolumeWeightedAveragePrice(df["high"], df["low"], df["close"], df["volume"]).volume_weighted_average_price()

    df = df.dropna().reset_index(drop=True)
    return df

# ================================
# CREATE TARGET
# ================================
def create_target(df):
    df["future_price"] = df["close"].shift(-TARGET_LOOKAHEAD)
    df["target"] = (df["future_price"] > df["close"]).astype(int)
    df = df.dropna().reset_index(drop=True)
    return df

# ================================
# TRAIN MODEL WITH CALIBRATION
# ================================
def train_model(df):
    X = df[FEATURES]
    y = df["target"]

    split = int(len(df) * 0.8)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    base_model = XGBClassifier(
        n_estimators=600,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=42
    )

    model = CalibratedClassifierCV(base_model, method='isotonic', cv=3)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print("Model Accuracy:", acc)

    return model

# ================================
# PREDICT NEXT MOVE
# ================================
def predict(model, df):
    latest = df[FEATURES].iloc[[-1]]
    pred = model.predict(latest)[0]
    prob = model.predict_proba(latest)[0]

    direction = "UP" if pred == 1 else "DOWN"
    confidence = prob.max()
    current_price = df["close"].iloc[-1]

    print(f"Current Price: {current_price}")
    print(f"Prediction: {direction}, Confidence: {confidence:.3f}")
    print("Time:", datetime.utcnow())
    return direction, confidence

# ================================
# WAIT UNTIL NEXT 1-MIN INTERVAL
# ================================
def sleep_to_next_1min():
    now = datetime.utcnow()
    seconds_since = now.second
    sleep_time = 60 - seconds_since
    if sleep_time < 0:
        sleep_time = 0
    print(f"Sleeping {sleep_time} seconds")
    time.sleep(sleep_time)

# ================================
# MAIN LOOP
# ================================
print("Starting Crypto ML Engine (1-min retraining)")

# Initial full fetch
df = fetch_data()
df = build_features(df)
df = create_target(df)

while True:
    try:
        # Incremental update
        latest = fetch_latest_candle()
        df = pd.concat([df, latest], ignore_index=True).tail(CANDLE_LIMIT)
        df = build_features(df)
        df = create_target(df)

        print("\nTraining model...")
        model = train_model(df)

        print("Predicting...")
        predict(model, df)

        sleep_to_next_1min()

    except Exception as e:
        print("Error:", e)
        time.sleep(10)

/Users/temuulenbadarchoutlook.com/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Starting Crypto ML Engine (1-min retraining)

Training model...
Model Accuracy: 0.45789473684210524
Predicting...
Current Price: 67905.09
Prediction: UP, Confidence: 0.625
Time: 2026-03-07 06:45:44.129205
Sleeping 16 seconds

Training model...
Model Accuracy: 0.4486486486486487
Predicting...
Current Price: 67893.14
Prediction: UP, Confidence: 0.627
Time: 2026-03-07 06:46:05.572372
Sleeping 55 seconds

Training model...
Model Accuracy: 0.4388888888888889
Predicting...
Current Price: 67877.03
Prediction: UP, Confidence: 0.668
Time: 2026-03-07 06:47:08.258335
Sleeping 52 seconds


KeyboardInterrupt: 